In [53]:
"""Table: Logs

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| num         | varchar |
+-------------+---------+
In SQL, id is the primary key for this table.
id is an autoincrement column starting from 1.
 

Find all numbers that appear at least three times consecutively.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Logs table:
+----+-----+
| id | num |
+----+-----+
| 1  | 1   |
| 2  | 1   |
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |
+----+-----+
Output: 
+-----------------+
| ConsecutiveNums |
+-----------------+
| 1               |
+-----------------+
Explanation: 1 is the only number that appears consecutively for at least three times."""

'Table: Logs\n\n+-------------+---------+\n| Column Name | Type    |\n+-------------+---------+\n| id          | int     |\n| num         | varchar |\n+-------------+---------+\nIn SQL, id is the primary key for this table.\nid is an autoincrement column starting from 1.\n \n\nFind all numbers that appear at least three times consecutively.\n\nReturn the result table in any order.\n\nThe result format is in the following example.\n\n \n\nExample 1:\n\nInput: \nLogs table:\n+----+-----+\n| id | num |\n+----+-----+\n| 1  | 1   |\n| 2  | 1   |\n| 3  | 1   |\n| 4  | 2   |\n| 5  | 1   |\n| 6  | 2   |\n| 7  | 2   |\n+----+-----+\nOutput: \n+-----------------+\n| ConsecutiveNums |\n+-----------------+\n| 1               |\n+-----------------+\nExplanation: 1 is the only number that appears consecutively for at least three times.'

In [54]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.getOrCreate()

logs_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("num", StringType(), True),
])

logs_data = [
    (1, "1"),
    (2, "1"),
    (3, "1"),
    (4, "2"),
    (5, "1"),
    (6, "2"),
    (7, "2"),
]

logs_df = spark.createDataFrame(logs_data, schema=logs_schema)


In [55]:
logs_df.show()

+---+---+
| id|num|
+---+---+
|  1|  1|
|  2|  1|
|  3|  1|
|  4|  2|
|  5|  1|
|  6|  2|
|  7|  2|
+---+---+



In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext

# --------------------------------------------------
# Job arguments
# --------------------------------------------------
args = getResolvedOptions(
    sys.argv,
    [
        "JOB_NAME",
        "S3_BASE_PATH"
    ]
)

# --------------------------------------------------
# Glue Context ONLY (no custom SparkSession)
# --------------------------------------------------
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
logger = glueContext.get_logger()

logger.info("Starting Oracle TEST export (10 rows per table)")

# --------------------------------------------------
# Oracle JDBC URL (YES, still required)
# --------------------------------------------------
ORACLE_JDBC_URL = (
    "jdbc:oracle:thin:@(DESCRIPTION="
    "(LOAD_BALANCE=off)"
    "(FAILOVER=on)"
    "(ADDRESS=(PROTOCOL=TCP)(HOST=AE-P-ORA-RPTP-PRIM-V.construction.com)(PORT=1527))"
    "(ADDRESS=(PROTOCOL=TCP)(HOST=AE-P-ORA-RPTP-STBY-V.construction.com)(PORT=1527))"
    "(CONNECT_DATA=(SERVER=DEDICATED)(SERVICE_NAME=rptp_dg))"
    ")"
)

# --------------------------------------------------
# Step 1: Discover tables
# --------------------------------------------------
tables_dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="oracle",
    connection_options={
        "connectionName": "AE-P-ORA-RPTP-PRIM-V connection",
        "query": """
            SELECT owner, table_name
            FROM dba_tables
            WHERE owner = 'EFEED'
               OR (owner = 'DRAIN' AND table_name = 'RECORD_PROJECT_DR')
        """
    }
)

tables = tables_dyf.toDF().collect()
logger.info(f"Found {len(tables)} tables")

# --------------------------------------------------
# Step 2: Read & write 10 rows per table
# --------------------------------------------------
for row in tables:
    owner = row["OWNER"]
    table = row["TABLE_NAME"]
    full_table_name = f"{owner}.{table}"

    output_path = f"{args['S3_BASE_PATH']}/{owner}/{table}/test_from_glue/"

    logger.info(f"Exporting {full_table_name}")

    data_dyf = glueContext.create_dynamic_frame.from_options(
        connection_type="oracle",
        connection_options={
            "connectionName": "AE-P-ORA-RPTP-PRIM-V connection",
            "query": f"""
                SELECT *
                FROM {full_table_name}
                WHERE ROWNUM <= 10
            """
        }
    )

    (
        data_dyf
        .toDF()
        .write
        .mode("overwrite")
        .csv(output_path)
    )

    logger.info(f"Completed {full_table_name}")

logger.info("Oracle TEST export completed")


In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext

# --------------------------------------------------
# Job arguments
# --------------------------------------------------
args = getResolvedOptions(
    sys.argv,
    [
        "JOB_NAME",
        "S3_BASE_PATH"
    ]
)

# --------------------------------------------------
# Glue Context ONLY (no custom SparkSession)
# --------------------------------------------------
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
logger = glueContext.get_logger()

logger.info("Starting Oracle TEST export (10 rows per table)")

# --------------------------------------------------
# Oracle JDBC URL (YES, still required)
# --------------------------------------------------
# ORACLE_JDBC_URL = (
#     "jdbc:oracle:thin:@(DESCRIPTION="
#     "(LOAD_BALANCE=off)"
#     "(FAILOVER=on)"
#     "(ADDRESS=(PROTOCOL=TCP)(HOST=AE-P-ORA-RPTP-PRIM-V.construction.com)(PORT=1527))"
#     "(ADDRESS=(PROTOCOL=TCP)(HOST=AE-P-ORA-RPTP-STBY-V.construction.com)(PORT=1527))"
#     "(CONNECT_DATA=(SERVER=DEDICATED)(SERVICE_NAME=rptp_dg))"
#     ")"
# )
oracle_connection_name='AE-P-ORA-RPTP-PRIM-V connection'
oracle_table_name='EFEED.AUD_FPR_FEED_PROJECT'
import traceback

logger = glueContext.get_logger()

logger.info("Starting Oracle connection test...")

try:
    oracle_dynamic_frame = glueContext.create_dynamic_frame.from_options(
        connection_type="jdbc",   # IMPORTANT
        connection_options={
            "useConnectionProperties": "true",
            "connectionName": oracle_connection_name,
            "dbtable": oracle_table_name
        }
    )

    # Force execution (this is what really tests the connection)
    row_count = oracle_dynamic_frame.count()

    logger.info(
        f"SUCCESS: Oracle connection established. "
        f"Table={oracle_table_name}, Rows={row_count}"
    )

except Exception as e:
    logger.error("FAILED: Oracle connection test failed")
    logger.error(str(e))
    logger.error(traceback.format_exc())
    raise
# # --------------------------------------------------
# # Step 1: Discover tables
# # --------------------------------------------------
# tables_dyf = glueContext.create_dynamic_frame.from_options(
#     connection_type="oracle",
#     connection_options={
#         "connectionName": "AE-P-ORA-RPTP-PRIM-V connection",
#         "query": """
#             SELECT owner, table_name
#             FROM dba_tables
#             WHERE owner = 'EFEED'
#               OR (owner = 'DRAIN' AND table_name = 'RECORD_PROJECT_DR')
#         """
#     }
# )

# tables = tables_dyf.toDF().collect()
# logger.info(f"Found {len(tables)} tables")

# # --------------------------------------------------
# # Step 2: Read & write 10 rows per table
# # --------------------------------------------------
# for row in tables:
#     owner = row["OWNER"]
#     table = row["TABLE_NAME"]
#     full_table_name = f"{owner}.{table}"

#     output_path = f"{args['S3_BASE_PATH']}/{owner}/{table}/test_from_glue/"

#     logger.info(f"Exporting {full_table_name}")

#     data_dyf = glueContext.create_dynamic_frame.from_options(
#         connection_type="oracle",
#         connection_options={
#             "connectionName": "AE-P-ORA-RPTP-PRIM-V connection",
#             "query": f"""
#                 SELECT *
#                 FROM {full_table_name}
#                 WHERE ROWNUM <= 10
#             """
#         }
#     )

#     (
#         data_dyf
#         .toDF()
#         .write
#         .mode("overwrite")
#         .csv(output_path)
#     )



logger.info("Oracle TEST export completed")


In [ ]:
import sys
import traceback
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext

# --------------------------------------------------
# Job arguments
# --------------------------------------------------
args = getResolvedOptions(
    sys.argv,
    [
        "JOB_NAME",
        "S3_BASE_PATH"
    ]
)

# --------------------------------------------------
# Glue Context ONLY
# --------------------------------------------------
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
logger = glueContext.get_logger()

# --------------------------------------------------
# Constants
# --------------------------------------------------
ORACLE_CONNECTION_NAME = "AE-P-ORA-RPTP-PRIM-V connection"

logger.info("Starting Oracle TEST export (10 rows per table)")

# --------------------------------------------------
# Step 1: Discover tables
# --------------------------------------------------
try:
    logger.info("Discovering Oracle tables...")

    tables_dyf = glueContext.create_dynamic_frame.from_options(
        connection_type="jdbc",   # IMPORTANT: use jdbc
        connection_options={
            "useConnectionProperties": "true",
            "connectionName": ORACLE_CONNECTION_NAME,
            "query": """
                SELECT owner, table_name
                FROM dba_tables
                WHERE owner = 'EFEED'
                   OR (owner = 'DRAIN' AND table_name = 'RECORD_PROJECT_DR')
            """
        }
    )

    tables = tables_dyf.toDF().collect()
    logger.info(f"Found {len(tables)} tables")

except Exception as e:
    logger.error("Failed while discovering tables")
    logger.error(str(e))
    logger.error(traceback.format_exc())
    raise

# --------------------------------------------------
# Step 2: Read & export 10 rows per table
# --------------------------------------------------
for row in tables:
    owner = row["OWNER"]
    table = row["TABLE_NAME"]
    full_table_name = f"{owner}.{table}"

    output_path = (
        f"{args['S3_BASE_PATH']}/{owner}/{table}/test_from_glue/"
    )

    logger.info(f"Exporting 10 rows from {full_table_name}")

    try:
        data_dyf = glueContext.create_dynamic_frame.from_options(
            connection_type="jdbc",   # IMPORTANT
            connection_options={
                "useConnectionProperties": "true",
                "connectionName": ORACLE_CONNECTION_NAME,
                "dbtable": f"""
                    (
                        SELECT *
                        FROM {full_table_name}
                        WHERE ROWNUM <= 10
                    )
                """
            }
        )

        (
            data_dyf
            .toDF()
            .write
            .mode("overwrite")
            .option("header", "true")
            .csv(output_path)
        )

        logger.info(f"Completed export for {full_table_name}")

    except Exception as e:
        logger.error(f"Failed exporting {full_table_name}")
        logger.error(str(e))
        logger.error(traceback.format_exc())
        raise

logger.info("Oracle TEST export completed successfully")
